In [1]:
!pip install pandas
!pip install nltk
!pip install scikit-learn

In [2]:
import pandas as pd

# Load the pre-split train and test datasets
df_train = pd.read_json("train.json")
df_test = pd.read_json("test.json")

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

Train shape: (25000, 2)
Test shape:  (25000, 2)


In [3]:
print("=== TRAIN SAMPLE ===")
print(df_train.head(3))
print("\n=== LABEL DISTRIBUTION (TRAIN) ===")
print(df_train["label"].value_counts())
print("\n=== LABEL DISTRIBUTION (TEST) ===")
print(df_test["label"].value_counts())

=== TRAIN SAMPLE ===
                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1

=== LABEL DISTRIBUTION (TRAIN) ===
label
1    12500
0    12500
Name: count, dtype: int64

=== LABEL DISTRIBUTION (TEST) ===
label
1    12500
0    12500
Name: count, dtype: int64


In [4]:
# Check and remove duplicates
print("Train duplicates:", df_train.duplicated().sum())
print("Test duplicates:", df_test.duplicated().sum())
df_train = df_train.drop_duplicates()
df_test = df_test.drop_duplicates()
print(f"After dedup — Train: {len(df_train)}, Test: {len(df_test)}")

Train duplicates: 96
Test duplicates: 199
After dedup — Train: 24904, Test: 24801


In [5]:
# NLTK no longer needed — preprocessing is handled by TfidfVectorizer's built-in options.
print("Skipping NLTK — using pure sklearn pipeline.")

Skipping NLTK — using pure sklearn pipeline.


In [6]:
# No custom preprocessor needed — TfidfVectorizer handles lowercasing and stop words natively.

In [7]:
# Raw text flows directly into the pipeline — no manual preprocessing step.
print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (24904, 2)
Test shape: (24801, 2)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

# Pure sklearn pipeline — no custom functions, serializes cleanly with joblib.
pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=100000,
        sublinear_tf=True,
        min_df=2,
    )),
    ("classifier", LinearSVC(C=1.0, max_iter=2000)),
])

In [9]:
X_train, y_train = df_train["text"], df_train["label"]
X_test, y_test = df_test["text"], df_test["label"]

pipeline.fit(X_train, y_train)
best_model = pipeline
print("Training complete.")

Training complete.


In [10]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = best_model.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.8856

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.89      0.89     12361
           1       0.89      0.88      0.89     12440

    accuracy                           0.89     24801
   macro avg       0.89      0.89      0.89     24801
weighted avg       0.89      0.89      0.89     24801



In [11]:
# Show predictions on a few test samples (raw text — pipeline handles preprocessing)
sample_texts = df_test["text"].iloc[:3].tolist()
sample_labels = df_test["label"].iloc[:3].tolist()

predictions = best_model.predict(sample_texts)
scores = best_model.decision_function(sample_texts)

for i, text in enumerate(sample_texts):
    print(f"Text (truncated): {text[:80]}...")
    print(f"True: {sample_labels[i]}  |  Predicted: {predictions[i]}  |  Score: {scores[i]:.2f}")
    print("-" * 60)

Text (truncated): I went and saw this movie last night after being coaxed to by a few friends of m...
True: 1  |  Predicted: 1  |  Score: 0.31
------------------------------------------------------------
Text (truncated): Actor turned director Bill Paxton follows up his promising debut, the Gothic-hor...
True: 1  |  Predicted: 1  |  Score: 0.95
------------------------------------------------------------
Text (truncated): As a recreational golfer with some knowledge of the sport's history, I was pleas...
True: 1  |  Predicted: 1  |  Score: 0.61
------------------------------------------------------------


In [12]:
import joblib

model_filename = "network_anomaly_detection_model.joblib"
joblib.dump(best_model, model_filename)
print(f"Model saved to {model_filename}")

Model saved to network_anomaly_detection_model.joblib
